# Homework GraphFrames Part 2 Producer


In [1]:
import json
import sys
import time

sys.path.insert(0, "/home/hadoop/homework/spark-graphframe")

from kafka import KafkaProducer

from spark_graphframe_homework import (
    BOOTSTRAP_SERVERS,
    TOPIC_NAME,
    create_kafka_topic,
    ensure_data,
    iter_trip_messages,
    make_run_id,
    write_latest_run_id,
)


In [2]:
ensure_data(sync_hdfs=False)
create_kafka_topic(TOPIC_NAME)

RUN_ID = write_latest_run_id(make_run_id())
MESSAGE_LIMIT = 250  # Set to None to stream the full CSV.
SLEEP_SECONDS = 0.01

preview_messages = list(iter_trip_messages(limit=3, run_id=RUN_ID))
{"run_id": RUN_ID, "preview_messages": preview_messages}


{'run_id': 'trip-run-20260317T083151Z',
 'preview_messages': [{'run_id': 'trip-run-20260317T083151Z',
   'trip_id': 913460,
   'duration_seconds': 765,
   'start_date': '8/31/2015 23:26',
   'start_station': 'Harry Bridges Plaza (Ferry Building)',
   'src': 50,
   'end_date': '8/31/2015 23:39',
   'end_station': 'San Francisco Caltrain (Townsend at 4th)',
   'dst': 70,
   'bike_id': 288,
   'subscriber_type': 'Subscriber',
   'zip_code': '2139'},
  {'run_id': 'trip-run-20260317T083151Z',
   'trip_id': 913459,
   'duration_seconds': 1036,
   'start_date': '8/31/2015 23:11',
   'start_station': 'San Antonio Shopping Center',
   'src': 31,
   'end_date': '8/31/2015 23:28',
   'end_station': 'Mountain View City Hall',
   'dst': 27,
   'bike_id': 35,
   'subscriber_type': 'Subscriber',
   'zip_code': '95032'},
  {'run_id': 'trip-run-20260317T083151Z',
   'trip_id': 913455,
   'duration_seconds': 307,
   'start_date': '8/31/2015 23:13',
   'start_station': 'Post at Kearny',
   'src': 47,
   

In [3]:
producer = KafkaProducer(
    bootstrap_servers=[BOOTSTRAP_SERVERS],
    value_serializer=lambda payload: json.dumps(payload).encode("utf-8"),
    key_serializer=lambda key: str(key).encode("utf-8"),
    acks="all",
)


def stream_trip_rows(limit=MESSAGE_LIMIT, sleep_seconds=SLEEP_SECONDS, run_id=RUN_ID):
    sent = 0
    started = time.time()
    last_offset = None

    for message in iter_trip_messages(limit=limit, run_id=run_id):
        future = producer.send(TOPIC_NAME, key=message["trip_id"], value=message)
        metadata = future.get(timeout=30)
        last_offset = metadata.offset
        sent += 1
        if sleep_seconds:
            time.sleep(sleep_seconds)

    producer.flush()
    elapsed = round(time.time() - started, 2)
    return {
        "run_id": run_id,
        "sent_messages": sent,
        "elapsed_seconds": elapsed,
        "last_offset": last_offset,
    }


producer_result = stream_trip_rows()
producer.close()
producer_result


{'run_id': 'trip-run-20260317T083151Z',
 'sent_messages': 250,
 'elapsed_seconds': 3.61,
 'last_offset': 746}